# IOI in GPT-2 small - activation patching for circuit discovery

Step 4 of 6 in the mech interp curriculum.

We load GPT-2 small (a real pre-trained 124M-parameter transformer), confirm it solves the indirect-object identification (IOI) task, and use activation patching to find the specific attention heads responsible. The expected output is a 12×12 heatmap with a small cluster of bright cells in layers 9-10 - the name mover heads.

Read `README.md` first. It explains the task, the metric (logit diff), and what activation patching is.

Expected runtime on Colab T4:, most of which is downloading GPT-2 (~500 MB) and the patching sweep.

## 1. Setup

Install TransformerLens. The `-q` flag keeps the output tidy.

In [ ]:
%pip install -q transformer_lens

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

from transformer_lens import HookedTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_grad_enabled(False)   # we never train here; saves memory
print(f'Using device: {device}')

## 2. Load GPT-2 small

One line. TransformerLens reformats the weights into a clean, hook-friendly architecture. The first time you run this it downloads ~500 MB; afterwards it's cached on disk.

In [ ]:
model = HookedTransformer.from_pretrained('gpt2', device=device)
model.eval()
print(f'Loaded {model.cfg.model_name}: {model.cfg.n_layers} layers, '
      f'{model.cfg.n_heads} heads/layer, d_model={model.cfg.d_model}')

## 3. Build the IOI prompts

Each example consists of a clean prompt (the IOI task as written) and a corrupted prompt where IO and S are swapped. Same words, same token count, but the right answer flips:

- Clean   : `When John and Mary went to the store, John gave a drink to` → `" Mary"`
- Corrupt : `When Mary and John went to the store, Mary gave a drink to` → `" John"`

We always measure `logitdiff = logit(IOclean) - logit(S_clean)` (i.e. always with the clean labels). On clean prompts it's strongly positive (model prefers IO); on corrupt prompts it flips to strongly negative (model now prefers S, because S is the IO of the corrupt sentence).

Using name-swap rather than the ABC variant from the paper is a deliberate simplification - it gives the cleanest, biggest logit-diff swing for the patching experiment, and we don't need to worry about a third name being multi-token. The trade-off is that name-swap corrupts more signals at once than ABC; we still find the name movers cleanly, but a finer-grained analysis would prefer ABC.

In [ ]:
# (IO, S, template). For each, the corrupt prompt swaps IO and S throughout.
# All names below tokenise to a single GPT-2 token with a leading space; we assert this below.
examples = [
    ('Mary',  'John',  'When {A} and {B} went to the store, {A} gave a drink to'),
    ('John',  'Mary',  'When {A} and {B} went to the store, {A} gave a drink to'),
    ('Tom',   'James', 'When {A} and {B} went to the park, {A} gave a drink to'),
    ('James', 'Tom',   'When {A} and {B} went to the park, {A} gave a drink to'),
    ('Bob',   'Adam',  'When {A} and {B} went to the school, {A} gave a drink to'),
    ('Adam',  'Bob',   'When {A} and {B} went to the school, {A} gave a drink to'),
    ('Mike',  'Sam',   'When {A} and {B} went to the office, {A} gave a drink to'),
    ('Sam',   'Mike',  'When {A} and {B} went to the office, {A} gave a drink to'),
]

# For clean: A=S, B=IO. The sentence says 'When S and IO went..., S gave a drink to' — predicts IO.
# For corrupt: A=IO, B=S. The sentence says 'When IO and S went..., IO gave a drink to' — predicts S.
clean_prompts, corrupt_prompts, io_ids, s_ids = [], [], [], []
for IO, S, template in examples:
    clean   = template.format(A=S,  B=IO)   # IO appears second, S is the subject
    corrupt = template.format(A=IO, B=S)    # IO and S swapped
    clean_prompts.append(clean)
    corrupt_prompts.append(corrupt)
    io_ids.append(model.to_single_token(' ' + IO))   # asserts single-token
    s_ids.append( model.to_single_token(' ' + S))

print('CLEAN  :', clean_prompts[0], '  → expects " Mary"')
print('CORRUPT:', corrupt_prompts[0], '  → expects " John"')
print(f'IO="Mary" token id: {io_ids[0]}   S="John" token id: {s_ids[0]}')

## 4. Confirm GPT-2 small solves IOI on clean prompts

Tokenise both prompt sets and run the model. Compute the logit diff at the final position: `logit(IO) - logit(S)`. On clean prompts it should be solidly positive (model prefers IO). On corrupted prompts it should be solidly negative (the swap means S is now the IO of the corrupt sentence - model prefers it).

Important sanity check: both prompt sets must have the same sequence length, so that patching at position `-1` aligns to the same predict-next-token position in both.

In [ ]:
clean_tokens   = model.to_tokens(clean_prompts)    # (batch, seq)
corrupt_tokens = model.to_tokens(corrupt_prompts)
assert clean_tokens.shape == corrupt_tokens.shape, (
    f'token-length mismatch: clean {clean_tokens.shape} vs corrupt {corrupt_tokens.shape}. '
    'Pick a different set of names.'
)
print(f'Token shape (both): {tuple(clean_tokens.shape)}')

io_tensor = torch.tensor(io_ids, device=device)
s_tensor  = torch.tensor(s_ids,  device=device)

def logit_diff(logits, io_ids, s_ids):
    """logits: (batch, seq, d_vocab) -- we read off the final position."""
    last = logits[:, -1, :]                          # (batch, d_vocab)
    diff = last.gather(1, io_ids.unsqueeze(1)) - last.gather(1, s_ids.unsqueeze(1))
    return diff.mean().item()

clean_logits, clean_cache = model.run_with_cache(clean_tokens)
corrupt_logits = model(corrupt_tokens)

clean_diff   = logit_diff(clean_logits,   io_tensor, s_tensor)
corrupt_diff = logit_diff(corrupt_logits, io_tensor, s_tensor)

print(f'CLEAN   logit diff (IO − S): {clean_diff:+.3f}   (model prefers IO ✓)')
print(f'CORRUPT logit diff (IO − S): {corrupt_diff:+.3f}   (model now prefers S ✓)')

## 5. Setting up activation patching

TransformerLens exposes every internal activation by hook name. The activation we want to patch is the per-head output `z` at every layer, available as `"blocks.{L}.attn.hookz"`. Its shape is `(batch, seq, nheads, d_head)`.

We define a hook that, on the corrupted forward pass, replaces one specific head's `z` at the final position with the cached clean value. Then we run the corrupted prompts through the model with this hook and measure the resulting logit diff.

In [ ]:
def patch_head_z_at_last_pos(z, hook, layer, head, clean_cache):
    """Replace z[:, -1, head, :] with the clean cache's value at this layer."""
    z[:, -1, head, :] = clean_cache[f'blocks.{layer}.attn.hook_z'][:, -1, head, :]
    return z

def patched_logit_diff(layer, head):
    hook_fn = partial(patch_head_z_at_last_pos, layer=layer, head=head, clean_cache=clean_cache)
    patched_logits = model.run_with_hooks(
        corrupt_tokens,
        fwd_hooks=[(f'blocks.{layer}.attn.hook_z', hook_fn)],
    )
    return logit_diff(patched_logits, io_tensor, s_tensor)

# sanity check on a known-good name-mover head (L9H9 in the IOI paper)
sample = patched_logit_diff(9, 9)
print(f'After patching L9H9: logit diff = {sample:+.3f}  '
      f'(clean was {clean_diff:+.3f}, corrupt was {corrupt_diff:+.3f})')

## 6. Run the patching sweep over all 144 heads

Loop over `(layer, head)` for all `12 × 12 = 144` heads. For each, run a patched forward pass and compute the recovery fraction:

```
recovery = (patcheddiff − corruptdiff) / (cleandiff − corruptdiff)
```

- `0.0` → no recovery (the head didn't matter).
- `1.0` → full recovery (patching this head alone is as good as running the clean prompt).

This takes.

In [ ]:
n_layers, n_heads = model.cfg.n_layers, model.cfg.n_heads
recovery = np.zeros((n_layers, n_heads))

for L in range(n_layers):
    for H in range(n_heads):
        diff = patched_logit_diff(L, H)
        recovery[L, H] = (diff - corrupt_diff) / (clean_diff - corrupt_diff)
    print(f'finished layer {L:2d}   max recovery in this layer: {recovery[L].max():+.3f}')

## 7. The heatmap - which heads carry the IO information?

A few bright cells in middle-to-late layers (~9-10) should stand out clearly. Those are the name mover heads.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
vmax = max(0.05, np.abs(recovery).max())
im = ax.imshow(recovery, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
ax.set_xticks(range(n_heads));  ax.set_xlabel('head')
ax.set_yticks(range(n_layers)); ax.set_ylabel('layer')
for L in range(n_layers):
    for H in range(n_heads):
        if abs(recovery[L, H]) > 0.1:
            ax.text(H, L, f'{recovery[L, H]*100:.0f}', ha='center', va='center',
                    color='black', fontsize=8)
plt.colorbar(im, label='recovery fraction')
ax.set_title('Activation-patching recovery per head\n'
             '(red = patching this head restores the clean answer = causally important)')
plt.tight_layout()
plt.show()

# top 5 heads by recovery
flat = [(L, H, recovery[L, H]) for L in range(n_layers) for H in range(n_heads)]
flat.sort(key=lambda t: -t[2])
print('\nTop 5 heads by recovery:')
for rank, (L, H, r) in enumerate(flat[:5], 1):
    print(f'  {rank}. L{L}H{H}  recovery = {r*100:.1f}%')

## 8. Inspect the name movers' attention patterns

For the top-recovery heads, plot the attention from the final position on a clean prompt. We expect to see attention concentrated on the IO name token.

In [ ]:
top_heads = [(L, H) for L, H, _ in flat[:4]]

fig, axes = plt.subplots(1, len(top_heads), figsize=(4 * len(top_heads), 4))
if len(top_heads) == 1:
    axes = [axes]

# use the first clean prompt for the visualization
example_tokens = clean_tokens[0:1]
_, ex_cache    = model.run_with_cache(example_tokens)
token_strs     = model.to_str_tokens(example_tokens[0])

for ax, (L, H) in zip(axes, top_heads):
    pattern   = ex_cache[f'blocks.{L}.attn.hook_pattern'][0, H].cpu().numpy()
    final_row = pattern[-1, :]
    ax.bar(range(len(final_row)), final_row)
    ax.set_xticks(range(len(token_strs)))
    ax.set_xticklabels(token_strs, rotation=45, ha='right', fontsize=8)
    ax.set_title(f'L{L}H{H} — attention from final pos', fontsize=10)
    ax.set_ylabel('attention weight')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Where the top heads attend from the final position\n(clean prompt: {clean_prompts[0]!r})', y=1.04)
plt.tight_layout()
plt.show()

Look at the bars. For the name mover heads, the tallest bar should be over the IO name token (e.g. `Mary` in our first prompt). That's the head doing exactly what its name suggests: at the final position, it attends to the indirect object name and copies it forward.

## 9. Discussion

What you just did is the central move in modern mech interp on real models:

1. Hypothesise that the model implements IOI via a small subset of attention heads.
2. Define a clean/corrupted pair of prompts that isolates the behaviour.
3. Choose a metric (logit diff) that quantitatively measures the behaviour.
4. Sweep activation patching across components.
5. Read the heatmap to find causally important components.
6. Verify by looking at attention patterns of the implicated heads.

What we found: a small cluster of heads in layers 9-10 is causally responsible for most of the IOI behaviour. Those are the name mover heads. They attend from the final position to the IO name and copy it to the output.

What we did not find (deliberately - out of scope):

- The S-inhibition heads in layers 7-8 that make the name movers look at IO instead of S. They write information into the residual stream at the subject position, not the final position - so patching their final-position z (what we did) misses them. Finding them requires patching the queries of the name movers (path patching).
- The duplicate token heads in early layers that detect `John ... John` and start the cascade.
- The previous-token heads and induction heads doing supporting work - yes, the same induction heads from step 3 show up in this circuit too.

The Wang et al. paper traces all of these. We did the most striking slice - the part that makes the activation-patching technique clear.

Why this matters: this is what mech interp actually looks like on a real model. The methodology is general - pick a behaviour, set up clean/corrupted prompts, patch, read the heatmap, follow up on the bright cells. Every recent mech interp paper on a real LLM uses some version of this loop.

Onwards to step 5 (sparse autoencoders), where we automate the feature-finding instead of doing it one head at a time.